# MuleNet Model Trainer (Kaggle)

This notebook is designed to train the XGBoost, GNN, and Isolation Forest models for MuleNet using Kaggle's powerful compute environments.

## 🚀 Instructions:
1. In your local MuleNet repository, run `python package_for_kaggle.py`. This will create `mulenet_for_kaggle.zip`.
2. On the right side of this Kaggle notebook, click **Add Input** -> **Upload a Dataset**.
3. Upload the `mulenet_for_kaggle.zip` file, as well as your large `processed_paysim.csv` dataset (if you have one).
4. Update the `DATASET_PATH` and `CODE_ZIP_PATH` variables in the cell below to point to your uploaded files.
5. Click **Run All**!

In [ ]:
import os
import zipfile
import shutil

# ==========================================
# ⚙️ CONFIGURATION
# Update these paths based on what you named your Kaggle datasets!
# Typically, uploaded datasets are mounted at /kaggle/input/YOUR_DATASET_NAME/
# ==========================================

CODE_ZIP_PATH = "/kaggle/input/mulenet-code/mulenet_for_kaggle.zip"  # Path to your uploaded zip
DATASET_PATH = "/kaggle/input/paysim-dataset/processed_paysim.csv"  # Path to your uploaded CSV

WORKING_DIR = "/kaggle/working/MuleNet"


### Step 1: Extract the Codebase

In [ ]:
if os.path.exists(WORKING_DIR):
    shutil.rmtree(WORKING_DIR)
os.makedirs(WORKING_DIR, exist_ok=True)

print("Extracting MuleNet code...")
with zipfile.ZipFile(CODE_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(WORKING_DIR)
print("Extraction complete!")


### Step 2: Prepare the Dataset
The training scripts expect the dataset to be in `MuleNet/processed_paysim.csv`. We will copy or symlink the dataset you uploaded into this location.

In [ ]:
dest_csv = os.path.join(WORKING_DIR, "processed_paysim.csv")
if os.path.exists(DATASET_PATH):
    print(f"Copying dataset from {DATASET_PATH} to {dest_csv}...")
    shutil.copy2(DATASET_PATH, dest_csv)
    print("Dataset ready!")
else:
    print("⚠️ WARNING: DATASET_PATH not found. If you have a small dataset already bundled in the zip, you can ignore this.")


### Step 3: Install Dependencies

In [ ]:
!pip install xgboost scikit-learn networkx pandas numpy


### Step 4: Run the Training Pipeline
This script will train XGBoost, Graph Attention Networks, and Isolation Forests.

In [ ]:
import sys
sys.path.append(WORKING_DIR)
os.chdir(WORKING_DIR)

!python ml_service/training/retrain_models.py


### Step 5: Export the Trained Models
We will zip up the newly trained models so you can download them directly from Kaggle and place them back into your local `MuleNet/ml_service/models/` folder.

In [ ]:
models_dir = os.path.join(WORKING_DIR, "ml_service", "models")
output_zip = "/kaggle/working/trained_models_download.zip"

print(f"Zipping models from {models_dir}...")
shutil.make_archive(output_zip.replace('.zip', ''), 'zip', models_dir)
print(f"\n✅ Success! You can now download '{os.path.basename(output_zip)}' from the 'Output' section on the right.")
